# 📌 Crawl Video + Comment TikTok bằng Google Colab

## 1. Giới thiệu

## 2. Chuẩn bị môi trường

### 2.1. Tạo thư mục lưu trữ dữ liệu

In [1]:
import os

TIKTOK_DATA_DIR = "D:/UIT/SE365 - Materials/Assignment/BTTH/1/datasets/tiktok"
print("Output absolute path:", os.path.abspath(TIKTOK_DATA_DIR))

Output absolute path: D:\UIT\SE365 - Materials\Assignment\BTTH\1\datasets\tiktok


### 2.2. Cài đặt thư viện cần thiết

Chúng ta cần:
* `yt-dlp` → tải video từ TikTok.
* `selenium` + `webdriver-manager` → dự phòng khi cần tự động hóa trình duyệt.

Chạy lệnh **`pip install yt-dlp selenium webdriver-manager`** trên terminal để cài đặt các thư viện trên.

### 2.3. Chuẩn bị Cookie

Để crawl comment TikTok, bạn cần file **`cookies.txt`** (xuất từ trình duyệt khi đang đăng nhập TikTok).

* Vào [https://www.tiktok.com](https://www.tiktok.com) trên Chrome.
* Cài extension **Get cookies.txt**.
* Xuất cookie TikTok và tải về máy thành `cookies.txt`.
* Upload file đó lên Colab.

In [2]:
with open("tiktok_cookies.txt", "r") as f:
    # Extract the actual cookie string, assuming it's the last non-empty line and doesn't start with #
    lines = f.readlines()
    COOKIE = ""

    # Start from the end to find the last valid cookie line
    for line in reversed(lines): 
        line = line.strip()
        if line and not line.startswith("#"):   # Adjustment based on the actual cookies.txt format (take a look at tiktok_cookies.txt to confirm)
            # For this case, let's assume the last non-comment line is the cookie value.
            COOKIE = line
            break

if not COOKIE:
    print("Warning: Could not extract a valid cookie string from cookies.txt. Comment crawling may fail.")

print("Cookie (first 50 chars):", COOKIE[:50] + "..." if len(COOKIE) > 50 else COOKIE)

Cookie (first 50 chars): www.tiktok.com	FALSE	/	FALSE	1785459161	msToken	hQ...


## 3. Thực hiện crawl data

In [3]:
import os
import requests
import time
import json
import yt_dlp
from urllib.parse import urlparse
from datetime import datetime, timezone

In [4]:
# Video mẫu để test
video_url = "https://www.tiktok.com/@/video/7541610727442976002"
video_id = video_url.split("/")[-1]
print("Video ID:", video_id)

video_output_dir = f"{TIKTOK_DATA_DIR}/{video_id}"
print("Output directory for video and metadata:", os.path.abspath(video_output_dir))

Video ID: 7541610727442976002
Output directory for video and metadata: D:\UIT\SE365 - Materials\Assignment\BTTH\1\datasets\tiktok\7541610727442976002


### 3.1. Hàm tải video TikTok bằng yt-dlp

In [5]:
def download_tiktok_with_yt_dlp(video_url, out_dir, write_info_json=True, max_filesize=None):
    """
    Tải video TikTok và metadata (info json) bằng yt-dlp.
    Trả về dict metadata (yt-dlp info) nếu thành công.
    """
    os.makedirs(out_dir, exist_ok=True)
    # template: lưu theo id
    out_template = os.path.join(out_dir, '%(id)s.%(ext)s')
    ydl_opts = {
        'outtmpl': out_template,
        'format': 'best',
        'noplaylist': True,
    }
    if write_info_json:
        ydl_opts['writedescription'] = True  # mô tả
        ydl_opts['writesubtitles'] = False
        ydl_opts['writeinfojson'] = True

    if max_filesize:
        ydl_opts['max_filesize'] = max_filesize

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=True)

    # Nếu yt-dlp tạo file info json, đọc nó
    info_json_path = None
    if 'id' in info:
        candidate = os.path.join(out_dir, f"{info['id']}.info.json")
        if os.path.exists(candidate):
            info_json_path = candidate
    if info_json_path:
        with open(info_json_path, 'r', encoding='utf-8') as f:
            info_data = json.load(f)
    else:
        info_data = info
    return info_data

In [6]:
meta = download_tiktok_with_yt_dlp(video_url=video_url, out_dir=video_output_dir)
print(meta)

[TikTok] Extracting URL: https://www.tiktok.com/@/video/7541610727442976002
[TikTok] 7541610727442976002: Downloading webpage


[TikTok] Solving JS challenge using native Python implementation
[TikTok] 7541610727442976002: Downloading webpage with challenge cookie
[info] 7541610727442976002: Downloading 1 format(s): bytevc1_720p_999007-1
[info] Writing video description to: D:\UIT\SE365 - Materials\Assignment\BTTH\1\datasets\tiktok\7541610727442976002\7541610727442976002.description
[info] Writing video metadata as JSON to: D:\UIT\SE365 - Materials\Assignment\BTTH\1\datasets\tiktok\7541610727442976002\7541610727442976002.info.json
[download] D:\UIT\SE365 - Materials\Assignment\BTTH\1\datasets\tiktok\7541610727442976002\7541610727442976002.mp4 has already been downloaded
[download] 100% of    5.42MiB
{'id': '7541610727442976002', 'formats': [{'ext': 'mp4', 'vcodec': 'h264', 'acodec': 'aac', 'format_id': 'download', 'url': 'https://v16-webapp-prime.tiktok.com/video/tos/no1a/tos-no1a-ve-0068c001-no/o4BZIWBEFyukX9PIrQiiqJpaRQE3vXKAQBYdL/?a=1988&bti=ODszNWYuMDE6&&bt=1662&ft=-Csk_m7nPD12N~p_md-Uxg.FLY6e3wv25XcAp&mime

### 3.2. Lấy thông tin oEmbed

In [7]:
def fetch_oembed(video_url, timeout=10):
    """
    Lấy dữ liệu oEmbed (nếu trang hỗ trợ).
    Trả về dict hoặc None.
    """
    oembed_url = "https://www.tiktok.com/oembed"
    try:
        r = requests.get(oembed_url, params={'url': video_url}, timeout=timeout, headers={'User-Agent':'Mozilla/5.0'})
        if r.status_code == 200:
            return r.json()
        else:
            return None
    except Exception as e:
        print("oEmbed error:", e)
        return None

In [8]:
# Lấy dữ liệu oEmbed (nếu có)
print(fetch_oembed(video_url=video_url))

{'version': '1.0', 'type': 'video', 'title': 'Sau bao nhiêu năm mới có ngày hoà bình… #hochiminh #chientranh #hoabinh #a80 #vietnam ', 'author_url': 'https://www.tiktok.com/@yeuvietnam633', 'author_name': 'YÊU VIỆT NAM🇻🇳', 'width': '100%', 'height': '100%', 'html': '<blockquote class="tiktok-embed" cite="https://www.tiktok.com/@yeuvietnam633/video/7541610727442976002" data-video-id="7541610727442976002" data-embed-from="oembed" style="max-width:605px; min-width:325px;"> <section> <a target="_blank" title="@yeuvietnam633" href="https://www.tiktok.com/@yeuvietnam633?refer=embed">@yeuvietnam633</a> <p>Sau bao nhiêu năm mới có ngày hoà bình… <a title="hochiminh" target="_blank" href="https://www.tiktok.com/tag/hochiminh?refer=embed">#hochiminh</a> <a title="chientranh" target="_blank" href="https://www.tiktok.com/tag/chientranh?refer=embed">#chientranh</a> <a title="hoabinh" target="_blank" href="https://www.tiktok.com/tag/hoabinh?refer=embed">#hoabinh</a> <a title="a80" target="_blank" hr

### 3.3. Lấy comment và reply tương ứng của video

#### 3.3.1. Lấy comment

In [9]:
def fetch_all_comments(aweme_id, cookie, max_comments=500):
    '''
    aweme_id: ID của video TikTok (có thể lấy từ URL hoặc metadata)
    cookie: Chuỗi cookie đã được làm sạch (cleaned cookie string)
    max_comments: Số lượng bình luận tối đa cần lấy
    '''
    
    url = "https://www.tiktok.com/api/comment/list/"
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": f"https://www.tiktok.com/@/video/{aweme_id}",
        "Cookie": cookie # Use the cleaned cookie string
    }

    cursor = 0
    all_comments = []

    while True:
        params = {
            "aid": 1988,
            "aweme_id": aweme_id,
            "cursor": cursor,
            "count": 20
        }
        r = requests.get(url, params=params, headers=headers)
        data = r.json()

        comments = data.get("comments", [])
        all_comments.extend(comments)

        print(f"Fetched {len(all_comments)} comments so far...")

        if not data.get("has_more"):
            break
        cursor = data.get("cursor", cursor + 20)  # update con trỏ

        if len(all_comments) >= max_comments:
            break

        time.sleep(1)  # tránh bị rate limit

    return all_comments

In [10]:
# Print sample comments
comments = fetch_all_comments(aweme_id=video_id, cookie=COOKIE, max_comments=200)
print("Total comments:", len(comments))
if comments:
    print(comments[0])
else:
    print("No comments fetched.")

Fetched 20 comments so far...
Fetched 39 comments so far...
Fetched 59 comments so far...
Fetched 79 comments so far...
Fetched 98 comments so far...
Fetched 118 comments so far...
Fetched 138 comments so far...
Fetched 158 comments so far...
Fetched 178 comments so far...
Fetched 198 comments so far...
Fetched 218 comments so far...
Total comments: 218
{'allow_download_photo': False, 'author_pin': False, 'aweme_id': '7541610727442976002', 'cid': '7542422658898969352', 'collect_stat': 0, 'comment_language': 'vi', 'comment_post_item_ids': None, 'create_time': 1756107130, 'digg_count': 292, 'fold_status': -1, 'image_list': None, 'is_author_digged': False, 'is_comment_translatable': True, 'is_high_purchase_intent': False, 'label_list': None, 'no_show': False, 'reply_comment': None, 'reply_comment_total': 5, 'reply_id': '0', 'reply_to_reply_id': '0', 'share_info': {'acl': {'code': 0, 'extra': '{}'}, 'desc': '𝙏𝙞𝙣𝙚_’s comment: Xin đừng gọi anh là liệt sĩ vô danh\nHãy gọi anh là người con yêu

In [11]:
# Print sample specific comment
comments[0]

{'allow_download_photo': False,
 'author_pin': False,
 'aweme_id': '7541610727442976002',
 'cid': '7542422658898969352',
 'collect_stat': 0,
 'comment_language': 'vi',
 'comment_post_item_ids': None,
 'create_time': 1756107130,
 'digg_count': 292,
 'fold_status': -1,
 'image_list': None,
 'is_author_digged': False,
 'is_comment_translatable': True,
 'is_high_purchase_intent': False,
 'label_list': None,
 'no_show': False,
 'reply_comment': None,
 'reply_comment_total': 5,
 'reply_id': '0',
 'reply_to_reply_id': '0',
 'share_info': {'acl': {'code': 0, 'extra': '{}'},
  'desc': '𝙏𝙞𝙣𝙚_’s comment: Xin đừng gọi anh là liệt sĩ vô danh\nHãy gọi anh là người con yêu nước\nNơi anh nằm có đất mẹ ôm ấp\nNay hoà bình tổ quốc nhớ công anh',
  'title': 'Sau bao nhiêu năm mới có ngày hoà bình… #hochiminh #chientranh #hoabinh #a80 #vietnam ',
  'url': 'https://t.tiktok.com/i18n/share/video/7541610727442976002/?_d=0&comment_author_id=7471132226919629832&mid=7533839243718740752&preview_pb=0&region=VN&sh

#### 3.3.2. Lấy reply của từng comment

In [12]:
def fetch_replies(aweme_id, comment_id, cookie, max_replies=100):
    '''
    aweme_id: ID của video TikTok
    comment_id: ID của comment cha
    cookie: Chuỗi cookie đã được làm sạch (cleaned cookie string)
    max_replies: Số lượng reply tối đa cần lấy
    '''
    
    url = "https://www.tiktok.com/api/comment/list/reply/"
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": f"https://www.tiktok.com/@/video/{aweme_id}",
        "Cookie": cookie
    }
    cursor = 0
    replies = []

    while True:
        params = {
            "aid": 1988,
            "aweme_id": aweme_id,
            "comment_id": comment_id,
            "cursor": cursor,
            "count": 20
        }
        r = requests.get(url, params=params, headers=headers)
        data = r.json()

        rs = data.get("comments", [])
        replies.extend(rs)

        if not data.get("has_more"):
            break
        cursor = data.get("cursor", cursor + 20)

        if len(replies) >= max_replies:
            break

        time.sleep(1)

    return replies

#### 3.3.3 Gộp hai hàm lấy comment và reply

In [13]:
def fetch_all_comments_with_replies(aweme_id, cookie, max_comments=200):
    '''
    aweme_id: ID của video TikTok
    cookie: Chuỗi cookie đã được làm sạch (cleaned cookie string)
    max_comments: Số lượng bình luận tối đa cần lấy
    '''

    # Dùng hàm fetch_all_comments() ở trên để lấy comment cha
    comments = fetch_all_comments(aweme_id, cookie, max_comments=max_comments)

    # Với mỗi comment cha, dùng fetch_replies() ở trên để lấy replies và gắn vào metadata của comment cha
    results = []
    for c in comments:
        item = {    # Thêm metadata cơ bản cho comment cha
            "cid": c.get("cid"),
            "text": c.get("text"),
            "author": c.get("user", {}).get("nickname"),
            "replies": []   # Khởi tạo list rỗng "replies" trong item để chứa replies
        }

        try:
            rs = fetch_replies(aweme_id, c["cid"], cookie, max_replies=50) # Gọi hàm lấy replies
            item["replies"] = [r for r in rs]   # Thêm replies vào item["replies"]
        except Exception as e:
            print(f"⚠️ Error fetching replies for {c.get('cid')}: {e}")

        results.append(item)    # Thêm item (metadata cơ bản cho comment cha và replies) vào results

    # Return JSON results: list các comment cha, mỗi comment cha có metadata cơ bản và list replies (nếu có)
    return results

In [14]:
data = fetch_all_comments_with_replies(aweme_id=video_id, cookie=COOKIE, max_comments=30)
print("Total comments with replies:", len(data))
print(json.dumps(data[0], ensure_ascii=False, indent=2))

Fetched 20 comments so far...
Fetched 39 comments so far...
Total comments with replies: 39
{
  "cid": "7542422658898969352",
  "text": "Xin đừng gọi anh là liệt sĩ vô danh\nHãy gọi anh là người con yêu nước\nNơi anh nằm có đất mẹ ôm ấp\nNay hoà bình tổ quốc nhớ công anh",
  "author": "𝙏𝙞𝙣𝙚_",
  "replies": [
    {
      "aweme_id": "7541610727442976002",
      "cid": "7545728496833856274",
      "collect_stat": 0,
      "comment_language": "un",
      "comment_post_item_ids": null,
      "create_time": 1756876833,
      "digg_count": 1,
      "fold_status": 0,
      "image_list": null,
      "is_author_digged": false,
      "is_comment_translatable": false,
      "is_high_purchase_intent": false,
      "label_list": null,
      "no_show": false,
      "reply_comment": null,
      "reply_id": "7542422658898969352",
      "reply_to_reply_id": "0",
      "share_info": {
        "acl": {
          "code": 1,
          "extra": "{\"is_share_handler_failed\":\"1\"}"
        },
        "desc"

## 7. Hàm Crawl toàn bộ một video

In [15]:
def crawl_one_tiktok(video_url, video_out_dir, use_comments=False, max_comments=200, cookie=None):
    '''' 
    video_url: URL của video TikTok cần crawl
    out_dir: Thư mục để lưu video và metadata (nếu có)
    use_comments: Nếu True, sẽ cố gắng lấy comment và replies (cần cookie hợp lệ)
    max_comments: Số lượng comment tối đa cần lấy (nếu use_comments=True)
    cookie: Chuỗi cookie đã được làm sạch (cleaned cookie string, cần nếu use_comments=True)
    '''

    os.makedirs(video_out_dir, exist_ok=True)
    
    result = {
        "url": video_url,
        "crawled_at": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z")
    }

    # 1. oEmbed
    # Lưu vào result["oembed"] metadata VỀ POST (người đăng, title,...) từ oEmbed (nếu có)
    oembed = fetch_oembed(video_url)
    result["oembed"] = oembed 

    # 2. yt-dlp download
    # Lưu vào result["yt_dlp_info"] metadata VỀ VIDEO (format, resolution,...) và tải video nếu thành công, ngược lại lưu lỗi vào result["yt_dlp_error"]
    try:
        meta = download_tiktok_with_yt_dlp(video_url, video_out_dir)
        result["yt_dlp_info"] = meta
    except Exception as e:
        result["yt_dlp_error"] = str(e)

    # Lấy aweme_id
    aweme_id = None
    aweme_id = oembed["embed_product_id"]
    print("aweme_id:", aweme_id)

    # 3. Comments
    # Lưu vào result["comments"] list các comment cha, metada cơ bản và list replies (nếu có) nếu use_comments=True và cookie hợp lệ
    # Ngược lại để trống hoặc ghi lỗi vào result["comments_error"]
    cmts = fetch_all_comments_with_replies(aweme_id, COOKIE, max_comments=30)
    result["comments"] = cmts

    # 4. Save metadata JSON
    vid_id = aweme_id or "tiktok_" + datetime.datetime.utcnow().strftime("%Y%m%d%H%M%S")
    json_path = os.path.join(video_out_dir, f"{vid_id}_crawl.json")
    json_path = json_path.replace(os.sep, "/") 
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    return result, json_path

In [16]:
# Thay URL ở đây bằng URL video TikTok muốn crawl
video_url = "https://www.tiktok.com/@/video/7541610727442976002"  
video_id = video_url.split("/")[-1]
video_output_dir = f"{TIKTOK_DATA_DIR}/{video_id}"

print("Crawling video:", video_url, "id:", video_id)

Crawling video: https://www.tiktok.com/@/video/7541610727442976002 id: 7541610727442976002


In [17]:
res, jsonp = crawl_one_tiktok(video_url, video_output_dir, use_comments=True, max_comments=100)
print("Metadata saved to:", jsonp)
print("Keys:", list(res.keys()))
print("Comments sample:", res.get("comments", [])[:2])

[TikTok] Extracting URL: https://www.tiktok.com/@/video/7541610727442976002
[TikTok] 7541610727442976002: Downloading webpage


[TikTok] Solving JS challenge using native Python implementation
[TikTok] 7541610727442976002: Downloading webpage with challenge cookie
[info] 7541610727442976002: Downloading 1 format(s): bytevc1_720p_999007-1
[info] Writing video description to: D:\UIT\SE365 - Materials\Assignment\BTTH\1\datasets\tiktok\7541610727442976002\7541610727442976002.description
[info] Writing video metadata as JSON to: D:\UIT\SE365 - Materials\Assignment\BTTH\1\datasets\tiktok\7541610727442976002\7541610727442976002.info.json
[download] D:\UIT\SE365 - Materials\Assignment\BTTH\1\datasets\tiktok\7541610727442976002\7541610727442976002.mp4 has already been downloaded
[download] 100% of    5.42MiB
aweme_id: 7541610727442976002
Fetched 20 comments so far...
Fetched 39 comments so far...
Metadata saved to: D:/UIT/SE365 - Materials/Assignment/BTTH/1/datasets/tiktok/7541610727442976002/7541610727442976002_crawl.json
Keys: ['url', 'crawled_at', 'oembed', 'yt_dlp_info', 'comments']
Comments sample: [{'cid': '754242

## 8. Kết quả

Sau khi chạy xong:

* Video TikTok sẽ được tải về thư mục `TIKTOK_DATA_DIR`.
* Một file JSON chứa metadata + comments (ví dụ: `7541610727442976002_crawl.json`).
* Bạn có thể mở file JSON bằng VSCode hoặc Notepad để xem chi tiết.